# Running a script on Jupyter

**Subject:** Introduction to Problem Solving and Reasoning (IPSR)  
**Unit 1 (Python Basics), Topic 2**  
**How to use this notebook:** read each section, run every code cell (Shift + Enter), then change the examples and run them again. Try each exercise before you open its solution.

## 1. Learning goals

After this notebook you can:

* run, edit and re-run code cells, and tell code cells from Markdown cells
* use the keyboard shortcuts for running cells
* explain what the **kernel** is, and when to restart it
* use a few **magic commands**: `%who`, `%timeit`, `%%writefile`, `%run`
* run a separate `.py` file from a notebook, with `%run` and with `subprocess`
* explain what `if __name__ == "__main__":` does
* avoid the two classic mistakes: running cells out of order and trusting stale variables

## 2. What is Jupyter?

**Jupyter Notebook** is a web page where you mix text, code and results in one file (an `.ipynb` file).
A notebook is split into **cells**. You run one cell at a time and see the result right under it.
This is why Jupyter is popular for learning, data science and quick experiments.

Behind the page, a separate program called the **kernel** runs your Python code.
The kernel keeps all your variables in memory for as long as it is alive.

## 3. Cells and how to run them

There are two main cell types:

| Cell type | What it holds | How to switch |
|---|---|---|
| **Code** | Python code that the kernel runs | press `Y` in command mode |
| **Markdown** | Formatted text, like this paragraph | press `M` in command mode |

A cell is in one of two modes. **Edit mode** (green border): you type inside the cell, press `Esc` to leave.
**Command mode** (blue border): you act on whole cells, press `Enter` to start editing.

| Shortcut | What it does |
|---|---|
| `Shift + Enter` | run the cell and move to the next one |
| `Ctrl + Enter` | run the cell and stay on it |
| `Alt + Enter` | run the cell and insert a new cell below |
| `A` / `B` (command mode) | insert a cell **A**bove / **B**elow |
| `D`, `D` (command mode) | delete the cell |
| `Z` (command mode) | undo a cell delete |

Run the next cell. The result of the **last line** is shown automatically, even without `print`.

In [1]:
2 + 3

5

In [2]:
print("first line")
10 * 4          # only the last expression is displayed

first line


40

The number in `In [ ]` on the left shows the order in which cells were run. A `*` means the cell is still running.
Add a semicolon at the end of the last line to hide its value:

In [3]:
10 * 4;

## 4. The kernel remembers

Variables you create in one cell are visible in every other cell, in any order.

In [4]:
price = 250

In [5]:
print(price * 2)

500


The kernel menu has these useful actions:

* **Interrupt**: stop a cell that is running too long (for example an endless loop)
* **Restart**: throw away all variables and start fresh
* **Restart and Run All**: restart, then run every cell from top to bottom. This is the best test that a notebook really works.

### Saving
Jupyter saves automatically. Press `Ctrl + S` to save by hand. The **saved file** keeps your code and outputs, but **not** the kernel's memory.

## 5. Magic commands

Jupyter (strictly, IPython) adds special commands that start with `%` or `%%`. They are not Python.

* `%name` is a **line magic**. It works on the rest of the line.
* `%%name` is a **cell magic**. It works on the whole cell, and must be on the first line.

Start with two helpful ones.

In [6]:
%who

price	 


`%who` lists the variables the kernel knows about. You should see `price`. Now measure how long a line takes with `%timeit`:

In [7]:
%timeit -n 1000 -r 3 sum(range(100))

762 ns ± 170 ns per loop (mean ± std. dev. of 3 runs, 1,000 loops each)


`%timeit` runs the line many times and reports the average. Your time will differ from your neighbour's, that is normal.
`%lsmagic` lists all magics, and `?` after a name shows help, for example `sum?`.

### Reset a variable
`del` removes a name from memory. `%reset -f` removes everything (the `-f` means "do not ask").

In [8]:
temp = 99
del temp
%who

price	 


## 6. Writing a `.py` file from a notebook

A **script** is a text file ending in `.py` that holds Python code. Real programs live in scripts.
The magic `%%writefile` saves the cell text into a file. We first move into a temporary folder so that nothing is left behind.

In [9]:
import os, tempfile

old_dir = os.getcwd()                 # remember where we were
work = tempfile.TemporaryDirectory()  # a throw-away folder
os.chdir(work.name)
print("Working in a temporary folder now")

Working in a temporary folder now


In [10]:
%%writefile hello.py
print("Hello from hello.py")
name = "script"
print("name inside the script is", name)

Writing hello.py


In [11]:
print(os.listdir("."))

['hello.py']


## 7. Running a script with `%run`

`%run file.py` runs the file **inside your notebook kernel**. Afterwards, the script's variables are available in the notebook, as if you had typed the code yourself.

In [12]:
%run hello.py
print("In the notebook, name is now:", name)

Hello from hello.py
name inside the script is script
In the notebook, name is now: script


You can also pass **command line arguments**. The script reads them from `sys.argv`, a list where item 0 is the script name.

In [13]:
%%writefile add.py
import sys
a = int(sys.argv[1])
b = int(sys.argv[2])
print("sum =", a + b)

Writing add.py


In [14]:
%run add.py 7 5

sum = 12


## 8. Running a script with `subprocess`

`subprocess` starts a **brand new Python program**, separate from the kernel. The script gets its own memory, so its variables do **not** appear in your notebook.
This is closer to what happens when you type `python hello.py` in a terminal.

In [15]:
import subprocess, sys

result = subprocess.run([sys.executable, "hello.py"], capture_output=True, text=True)
print("exit code:", result.returncode)     # 0 means success
print("output:")
print(result.stdout)

exit code: 0
output:
Hello from hello.py
name inside the script is script



`sys.executable` is the path of the Python that runs this notebook, so we always use the same Python.
A line starting with `!` also runs a shell command, for example `!python hello.py`, but `subprocess` is more reliable inside programs.

| | `%run` | `subprocess` |
|---|---|---|
| Runs in | the notebook kernel | a new process |
| Script variables visible afterwards | yes | no |
| Script crash stops your notebook | it shows an error | no, you read `returncode` and `stderr` |
| Only in Jupyter | yes | works in any Python program |

## 9. What does `if __name__ == "__main__":` do?

Every Python file has a built-in variable `__name__`.

* When the file is **run directly**, `__name__` is `"__main__"`.
* When the file is **imported** by another file, `__name__` is the file's own name, such as `"tools"`.

So the block under `if __name__ == "__main__":` runs only when you run the file directly, and is skipped when someone imports it.
This lets one file serve as both a program and a library.

In [16]:
%%writefile tools.py
def square(n):
    return n * n

print("tools.py: __name__ is", __name__)

if __name__ == "__main__":
    print("Running tools.py directly, square(4) =", square(4))

Writing tools.py


In [17]:
%run tools.py

tools.py: __name__ is __main__
Running tools.py directly, square(4) = 16


In [18]:
import tools            # importing: the main block must NOT run
print(tools.square(5))

tools.py: __name__ is tools
25


## 10. Edge cases and common mistakes

### Mistake 1: running cells out of order
The kernel runs cells in the order **you click**, not the order they appear on the page.
Here is the same effect, written in one cell so you can see it safely. Think of each line as a different cell.

In [19]:
count = 1           # imagine this is cell A
count = count + 1   # cell B, run once: count is 2
count = count + 1   # cell B, run again by mistake: count is 3
print(count)

3


If you click cell B three times, `count` becomes 4, and the page still shows the old output of earlier runs.
Cells that change a variable using its old value (`count = count + 1`) are **not safe to re-run**.

### Mistake 2: stale variables
If you **delete** a cell, the variable it created is still alive in the kernel. Your notebook may seem to work today and fail tomorrow, after a restart.

In [20]:
gone = 5
del gone            # same as the kernel losing the variable after a restart
try:
    print(gone)
except NameError as e:
    print("NameError:", e)

NameError: name 'gone' is not defined


### Mistake 3: shadowing a name that a script also uses
`%run` copies the script's variables into the notebook. If the script has a variable called `name`, it **overwrites** your own `name`.

In [21]:
name = "my own value"
%run hello.py
print(name)         # replaced by the script's value

Hello from hello.py
name inside the script is script
script


### Mistake 4: an error does not stop the notebook from remembering
If a cell fails half way, the lines before the error **did** run, so their variables exist.

In [22]:
try:
    first = 1
    second = 1 / 0      # ZeroDivisionError
    third = 3
except ZeroDivisionError:
    pass
print("first" in dir(), "second" in dir(), "third" in dir())

True False False


### A good habit
Before you share a notebook, use **Kernel, Restart and Run All**. If it runs top to bottom without error, other people can run it too.

### Clean up
We created files inside a temporary folder. Remove the folder and go back to where we started.

In [23]:
import sys
sys.modules.pop("tools", None)
os.chdir(old_dir)
work.cleanup()
print("temporary folder exists:", os.path.exists(work.name))

temporary folder exists: False


## 11. Quick summary

| Idea | Remember |
|---|---|
| Cell types | Code (Python) and Markdown (text) |
| Run a cell | `Shift + Enter` (move on), `Ctrl + Enter` (stay) |
| Kernel | the program that runs your code and keeps variables |
| Restart and Run All | the best check that a notebook works |
| `%who`, `%timeit` | list variables, time a line |
| `%%writefile name` | save the cell text into a file |
| `%run file.py` | run a script in the kernel, variables stay |
| `subprocess.run([...])` | run a script in a separate process |
| `__name__ == "__main__"` | true only when the file is run directly |
| Mistakes | cells out of order, stale variables, names overwritten by `%run` |

## 12. Exercises

### Exercise 1

Write a short comment that lists the keyboard shortcut for each action: run and move on, run and stay, insert cell below, delete a cell. Then in code, print the result of `%who`-style listing by using `dir()` to show whether the name `price` exists: `"price" in dir()`.

*Hint:* `dir()` lists the names currently defined in the notebook.

In [24]:
# Write your answer here

<details>
<summary><b>Show solution 1</b></summary>

`dir()` gives the list of known names, so `"price" in dir()` is `True` once the variable exists.

```python
# Shift + Enter : run and move on
# Ctrl + Enter  : run and stay
# B (command mode) : insert cell below
# D, D (command mode) : delete the cell
price = 250
print("price" in dir())
```

</details>

### Exercise 2

Use `%%writefile` to create a script `greet.py` that prints `Hello, ` followed by the first command line argument. Run it with `%run greet.py Asha`. (Work in a temporary folder and clean up.)

*Hint:* `sys.argv[1]` is the first argument after the script name.

In [25]:
# Write your answer here

<details>
<summary><b>Show solution 2</b></summary>

In your own notebook you would write the file with `%%writefile greet.py` in a cell of its own, then `%run greet.py Asha` in the next cell. Here both steps are done from Python so the solution fits in one cell. Output: `Hello, Asha`.

```python
import os, tempfile
old = os.getcwd()
tmp = tempfile.TemporaryDirectory()
os.chdir(tmp.name)

with open("greet.py", "w") as f:       # same file that %%writefile would make
    f.write("import sys\nprint('Hello,', sys.argv[1])\n")

get_ipython().run_line_magic("run", "greet.py Asha")

os.chdir(old)
tmp.cleanup()
```

</details>

### Exercise 3

Write a script file with a function `double(n)` and a main block that prints `double(21)`. Run it with `subprocess`, and show that the main block runs. Then import it and show that the main block does **not** run.

*Hint:* Compare the output of `subprocess.run([sys.executable, file], capture_output=True, text=True)` with the output of `import`.

In [26]:
# Write your answer here

<details>
<summary><b>Show solution 3</b></summary>

The direct run prints `main block: 42`. The import prints nothing, because `__name__` is `"dbl"` there, not `"__main__"`.

```python
import os, sys, subprocess, tempfile, importlib

with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, "dbl.py")
    with open(path, "w") as f:
        f.write("def double(n):\n    return 2 * n\n\n"
                "if __name__ == '__main__':\n    print('main block:', double(21))\n")

    r = subprocess.run([sys.executable, path], capture_output=True, text=True)
    print("direct run prints:", r.stdout.strip())

    sys.path.insert(0, d)
    import dbl                      # nothing is printed by the import
    print("after import, double(5) =", dbl.double(5))
    sys.path.remove(d)
    sys.modules.pop("dbl", None)
```

</details>

### Exercise 4

Predict, then check: what do the last two lines print?

```python
x = 10
x = x * 2
x = x * 2
print(x)
```
If you run the cell containing only the middle line `x = x * 2` three times (after `x = 10`), what is `x`? Show it with a loop.

*Hint:* Each run multiplies by 2 again.

In [27]:
# Write your answer here

<details>
<summary><b>Show solution 4</b></summary>

The first part gives 40. Running the same multiply cell three times gives 80. This is why re-running cells that use their own old value is risky.

```python
x = 10
x = x * 2
x = x * 2
print(x)        # 40

x = 10
for run in range(3):    # three clicks on the same cell
    x = x * 2
print(x)        # 80
```

</details>

### Exercise 5

Use `%timeit` (with `-n 100 -r 3`) to compare `sum(range(1000))` with a loop that adds the numbers 0 to 999 one by one. Do not print the times in your answer, just run both and say in a comment which is faster.

*Hint:* Put each `%timeit` in its own cell.

In [28]:
# Write your answer here

<details>
<summary><b>Show solution 5</b></summary>

In a real notebook, add a second cell: `%%timeit -n 100 -r 3` followed by `total = 0`, `for i in range(1000): total += i`. The built-in `sum` is usually faster because its loop runs in fast compiled code. (The cell magic `%%timeit` times the whole cell.)

```python
%timeit -n 100 -r 3 sum(range(1000))
```

</details>

### Exercise 6

Show that `subprocess` does not share variables with the notebook. Set `secret = 42` here, then run a child Python with `python -c "print('secret' in dir())"` and print its output.

*Hint:* Use `[sys.executable, "-c", "print('secret' in dir())"]`.

In [29]:
# Write your answer here

<details>
<summary><b>Show solution 6</b></summary>

The notebook prints `True` and the child prints `False`. The child process has its own empty memory.

```python
import sys, subprocess
secret = 42
r = subprocess.run([sys.executable, "-c", "print('secret' in dir())"], capture_output=True, text=True)
print("in notebook:", "secret" in dir())
print("in child   :", r.stdout.strip())
```

</details>

## 13. Where to read more

* Software Carpentry, *Running and Quitting* (notebook): https://swcarpentry.github.io/python-novice-gapminder/01-run-quit.html
* Real Python, *Jupyter Notebook introduction* (handout): https://realpython.com/jupyter-notebook-introduction/